# 네이버 블로그 크롤러
Google Colab에서 `google-colab-selenium`을 사용한 간편한 네이버 블로그 크롤링

## 1. 필수 패키지 설치

`google-colab-selenium`은 Colab 환경에서 Chrome과 ChromeDriver를 자동으로 설정해줍니다.

In [ ]:
# 필수 패키지 설치
!pip install -q google-colab-selenium beautifulsoup4

print("✅ 설치 완료!")

## 2. 라이브러리 임포트

In [ ]:
import time
import re
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

print("✅ 라이브러리 임포트 완료!")

## 3. WebDriver 설정 함수

In [ ]:
def setup_driver():
    """
    Chrome WebDriver 설정 (google-colab-selenium 사용)
    """
    chrome_options = Options()
    
    # Chrome 옵션 설정
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-infobars')
    chrome_options.add_argument('--disable-popup-blocking')
    chrome_options.add_argument('--ignore-certificate-errors')
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # google-colab-selenium으로 자동 설정
    print("ChromeDriver 설정 중...")
    driver = gs.Chrome(options=chrome_options)
    print("✅ ChromeDriver 설정 완료!")
    
    return driver

print("✅ setup_driver 함수 정의 완료!")

## 4. 블로그 콘텐츠 추출 함수

In [ ]:
def extract_blog_content(url, wait_time=5):
    """
    네이버 블로그 콘텐츠 추출

    Args:
        url (str): 네이버 블로그 URL
        wait_time (int): 페이지 로딩 대기 시간 (초)

    Returns:
        str: 추출된 블로그 텍스트 내용
    """
    driver = None
    try:
        # 드라이버 설정
        driver = setup_driver()

        # 페이지 열기
        print(f"블로그 URL 접속 중: {url}")
        driver.get(url)
        time.sleep(wait_time)

        # iframe 찾기 및 전환
        print("iframe으로 전환 중...")
        iframe = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "mainFrame"))
        )
        driver.switch_to.frame(iframe)

        # 페이지 소스 가져오기
        source = driver.page_source
        html = BeautifulSoup(source, "html.parser")

        # 콘텐츠 추출
        print("콘텐츠 추출 중...")
        content_container = html.select("div.se-main-container")

        if not content_container:
            print("경고: 콘텐츠를 찾을 수 없습니다.")
            return ""

        # 텍스트만 추출
        content = ''.join(str(content_container))

        # HTML 태그 제거
        pattern1 = '<[^>]*>'
        content = re.sub(pattern=pattern1, repl='', string=content)

        # 불필요한 패턴 제거
        pattern2 = """[\n\n\n\n\n// flash 오류를 우회하기 위한 함수 추가\nfunction _flash_removeCallback() {}"""
        content = content.replace(pattern2, '')

        # 텍스트 정제
        content = content.replace('\n', ' ')
        content = content.replace('\u200b', '')  # Zero-width space 제거
        content = content.replace('&ZeroWidthSpace;', '')

        # 연속된 공백 제거
        content = re.sub(r'\s+', ' ', content).strip()

        print("✅ 콘텐츠 추출 완료!")
        return content

    except Exception as e:
        print(f"❌ 에러 발생: {str(e)}")
        import traceback
        traceback.print_exc()
        return ""

    finally:
        if driver:
            driver.quit()
            print("Chrome 종료")

print("✅ extract_blog_content 함수 정의 완료!")

## 5. 단일 블로그 크롤링

In [ ]:
# 크롤링할 블로그 URL
blog_url = "https://blog.naver.com/yminsong/224075787010"

# 콘텐츠 추출
print("=" * 80)
print("네이버 블로그 크롤링 시작")
print("=" * 80)

content = extract_blog_content(blog_url)

if content:
    print("\n" + "=" * 80)
    print("📝 추출된 콘텐츠:")
    print("=" * 80)
    print(content)
    print("\n" + "=" * 80)
    print(f"✅ 총 {len(content)}자 추출 성공!")
    print("=" * 80)
else:
    print("\n❌ 콘텐츠를 추출하지 못했습니다.")

## 6. 여러 URL 크롤링하기 (선택사항)

In [ ]:
# 여러 개의 블로그 URL을 크롤링하고 싶다면 아래 코드를 사용하세요
naver_urls = [
    "https://blog.naver.com/yminsong/224075787010",
    # 추가 URL을 여기에 입력
]

contents = []

for idx, url in enumerate(naver_urls, 1):
    print(f"\n[{idx}/{len(naver_urls)}] 처리 중: {url}")
    content = extract_blog_content(url)
    if content:
        contents.append({
            'url': url,
            'content': content,
            'length': len(content)
        })
        print(f"✅ 성공: {len(content)}자")
    else:
        print("❌ 실패")
    print("-" * 80)
    
    # 서버 부담 최소화를 위한 대기
    if idx < len(naver_urls):
        time.sleep(2)

# 결과 출력
print("\n" + "=" * 80)
print(f"✅ 총 {len(contents)}개의 블로그 글을 크롤링했습니다.")
print("=" * 80)

for idx, item in enumerate(contents, 1):
    print(f"\n[{idx}] {item['url']}")
    print(f"길이: {item['length']:,}자")
    print(f"내용 미리보기: {item['content'][:200]}...")

## 7. 결과를 CSV 파일로 저장 (선택사항)

In [ ]:
import pandas as pd

if contents:
    # DataFrame 생성
    df = pd.DataFrame(contents)
    
    # CSV 파일로 저장
    df.to_csv('naver_blog_contents.csv', index=False, encoding='utf-8-sig')
    print("✅ CSV 파일로 저장 완료: naver_blog_contents.csv")
    
    # 다운로드 (Colab)
    from google.colab import files
    files.download('naver_blog_contents.csv')
else:
    print("저장할 콘텐츠가 없습니다.")